# Observing Conditions — reference solution

The task: for each `(galaxy, target morphology)` row, name the ordered pair of
observing conditions that pushes a hidden classifier's probability for the
target as high as possible.

The hidden model cannot be queried. What *can* be done is to model its response
surface: a CNN reads the clean cutout, is told which morphology we are steering
toward, and predicts all 400 normalised responses at once. At inference we take
the argmax of that predicted surface.

Two choices carry most of the score:

* **Supervise on the whole surface.** Every training row ships all 400
  measured values, not just the index of the best pair. Regressing the surface
  gives ~400x the signal per row versus treating the row as 400-way
  classification with one hot label.
* **Split by galaxy.** One galaxy contributes up to seven rows, so a row-level
  validation split puts the same image on both sides and reports a score well
  above the leaderboard.

One trap worth naming: horizontal and vertical flips map every condition in the
catalog to itself and are safe augmentations, but a 90-degree rotation swaps
`trail_h` with `trail_v`. Rotating inputs without permuting the 400-vector
target corrupts the labels for those conditions.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from PIL import Image

DATA   = Path("dataset/public")
SEED   = 20260731
EPOCHS = 36
SEEDS  = 3
dev = torch.device("cuda" if torch.cuda.is_available() else
                   ("mps" if torch.backends.mps.is_available() else "cpu"))
print("device:", dev)

## Load the rows and the measured surfaces

In [ ]:
tr   = pd.read_csv(DATA / "train.csv")
te   = pd.read_csv(DATA / "test.csv")
resp = pd.read_csv(DATA / "train_response.csv").set_index("row_id").loc[tr["row_id"]]
Y    = resp.to_numpy(np.float32)              # [rows, 400] already in [0,1]
ncls = int(pd.read_csv(DATA / "classes.csv")["class_index"].max()) + 1

print(f"{len(tr)} train rows over {tr.image_id.nunique()} galaxies")
print(f"{len(te)} test rows over {te.image_id.nunique()} galaxies")
print("response matrix:", Y.shape)

### One image bank, not one tensor per row

Rows outnumber galaxies about 6:1. Materialising an image per row wastes most of
the memory budget, so images are stored once as uint8 and rows carry an index.

In [ ]:
ids  = sorted(set(tr.image_id) | set(te.image_id))
order = {i: k for k, i in enumerate(ids)}
bank = np.zeros((len(ids), 69, 69, 3), np.uint8)
for i, k in order.items():
    bank[k] = np.asarray(Image.open(DATA / "images" / f"{i}.png").convert("RGB"))

tr_img, te_img = tr.image_id.map(order).to_numpy(), te.image_id.map(order).to_numpy()
tr_tgt, te_tgt = tr.target_class.to_numpy(), te.target_class.to_numpy()

px   = bank.astype(np.float32) / 255.
MEAN = torch.tensor(px.mean((0,1,2))).view(1,3,1,1)
STD  = torch.tensor(px.std((0,1,2)) + 1e-6).view(1,3,1,1)
del px

def batch(idx):
    a = np.ascontiguousarray(bank[idx].transpose(0,3,1,2)).astype(np.float32) / 255.
    return ((torch.from_numpy(a) - MEAN) / STD).to(dev)

## Model: image trunk + target embedding -> 400 responses

In [ ]:
class Surrogate(nn.Module):
    def __init__(self, ncls, npair=400, w=48):
        super().__init__()
        def blk(i, o, pool=True):
            L = [nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                 nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU()]
            if pool: L.append(nn.MaxPool2d(2))
            return nn.Sequential(*L)
        self.trunk = nn.Sequential(blk(3,w), blk(w,2*w), blk(2*w,4*w),
                                   blk(4*w,4*w,pool=False),
                                   nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.emb  = nn.Embedding(ncls, 64)
        self.head = nn.Sequential(nn.Linear(4*w+64, 512), nn.ReLU(),
                                  nn.Dropout(0.1), nn.Linear(512, npair))
    def forward(self, x, t):
        return self.head(torch.cat([self.trunk(x), self.emb(t)], 1))

## Galaxy-grouped holdout

Grouping by `image_id` is not optional here — it is the difference between an
honest estimate and one inflated by seeing the same galaxy in training.

In [ ]:
rng   = np.random.RandomState(SEED)
uniq  = np.unique(tr_img)
hold  = set(rng.permutation(uniq)[: max(1, len(uniq)//6)].tolist())
isval = np.array([i in hold for i in tr_img])
print(f"holdout: {isval.sum()} rows / {len(hold)} galaxies")

In [ ]:
@torch.no_grad()
def infer(net, img, tgt, bs=256):
    net.eval()
    return np.concatenate([
        net(batch(img[i:i+bs]), torch.from_numpy(tgt[i:i+bs]).to(dev)).cpu().numpy()
        for i in range(0, len(img), bs)])

def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Surrogate(ncls).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1.5e-3, weight_decay=1e-4)
    fit, BS = np.where(~isval)[0], 128
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=2.5e-3, total_steps=EPOCHS*int(np.ceil(len(fit)/BS)))
    Yt = torch.from_numpy(Y)
    for ep in range(EPOCHS):
        net.train(); np.random.shuffle(fit)
        for i in range(0, len(fit), BS):
            b = fit[i:i+BS]
            x = batch(tr_img[b])
            if np.random.rand() < 0.5: x = torch.flip(x, dims=[3])   # safe
            if np.random.rand() < 0.5: x = torch.flip(x, dims=[2])   # safe
            p = net(x, torch.from_numpy(tr_tgt[b]).to(dev)); y = Yt[b].to(dev)
            loss = F.mse_loss(torch.sigmoid(p), y)*10.0 \
                 + F.kl_div(F.log_softmax(p,1), F.softmax(y*8,1), reduction="batchmean")
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    vi = np.where(isval)[0]
    v  = float(Y[vi, infer(net, tr_img[vi], tr_tgt[vi]).argmax(1)].mean())
    print(f"  seed {seed}: holdout {v:.4f}", flush=True)
    return net, v

## Train an ensemble and write the submission

In [ ]:
preds, vals = [], []
for s in range(SEEDS):
    net, v = train_one(SEED + s)
    vals.append(v)
    preds.append(infer(net, te_img, te_tgt))

pick = np.mean(preds, 0).argmax(1)
sub  = pd.DataFrame(dict(row_id=te.row_id, op1=pick//20, op2=pick%20))
Path("working").mkdir(exist_ok=True)
sub.to_csv("working/submission.csv", index=False)
print(f"mean galaxy-grouped holdout: {np.mean(vals):.4f}")
sub.head()

### What the score means

`1.0` would mean picking the single best of the 400 condition pairs for every
row; `0.0` the single worst. The cheap baseline to beat is a lookup table of the
best pair per `(true morphology, target morphology)` — anything at or below it
has learned the class-level prior and nothing about the individual galaxy.